In [1]:
!pip install trimesh
!pip install open3d
!pip install 'git+https://github.com/facebookresearch/pytorch3d.git@main'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 121.1 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


  Cloning https://github.com/facebookresearch/pytorch3d.git (to revision main) to /tmp/pip-req-build-bb_rbm_v
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/pytorch3d.git /tmp/pip-req-build-bb_rbm_v
  Resolved https://github.com/facebookresearch/pytorch3d.git to commit c8fcd83ff96fa0a5893c0b994f9285d7aa772540
  Preparing metadata (setup.py) ... done


In [2]:
import argparse
import csv
import os
import open3d as o3d
import numpy as np
import torch
import trimesh

from pytorch3d.io import load_obj
from pytorch3d.structures import Meshes
from pytorch3d.ops import sample_points_from_meshes, knn_points, iterative_closest_point

In [13]:
"""
eval_3d_metrics.py

Evalua la calidad 3D de reconstrucciones (voxel o mesh) contra un modelo
ground truth, calculando:

    - Chamfer Distance
    - F-Score a distintos umbrales
    - IoU volumétrico (ocupación 3D)

Requiere:
    pip install pytorch3d trimesh rtree numpy torch

"""
import gc
DEFAULT_THRESHOLDS = [0.01, 0.02, 0.05]  # fracción del diámetro tras normalizar a esfera unitaria
DEFAULT_N_POINTS = 10000
DEFAULT_VOXEL_RES = 64


# ---------------------------------------------------------------------------
# Utilidades de carga y normalización
# ---------------------------------------------------------------------------

def load_mesh_p3d(path, device):
    tm_mesh = trimesh.load(path, force="mesh", process=False)
    verts = torch.tensor(tm_mesh.vertices, dtype=torch.float32, device=device)
    faces = torch.tensor(tm_mesh.faces, dtype=torch.int64, device=device)

    return Meshes(verts=[verts], faces=[faces])


def normalize_mesh(mesh):
    """Centra en el origen y escala para que quepa en una esfera unitaria."""
    verts = mesh.verts_packed()
    center = verts.mean(dim=0, keepdim=True)
    verts = verts - center
    scale = verts.norm(dim=1).max()
    verts = verts / scale
    return mesh.update_padded(verts[None])


def mesh_to_trimesh(mesh):
    verts = mesh.verts_packed().detach().cpu().numpy()
    faces = mesh.faces_packed().detach().cpu().numpy()
    return trimesh.Trimesh(vertices=verts, faces=faces, process=False)


# ---------------------------------------------------------------------------
# Métricas
# ---------------------------------------------------------------------------

def chamfer_and_fscore(pred_pts, gt_pts, thresholds):
    """pred_pts, gt_pts: tensores (1, N, 3)."""
    d_pred_to_gt = knn_points(pred_pts, gt_pts).dists.sqrt().squeeze(-1).squeeze(0)  # (N,)
    d_gt_to_pred = knn_points(gt_pts, pred_pts).dists.sqrt().squeeze(-1).squeeze(0)  # (N,)

    chamfer = d_pred_to_gt.mean().item() + d_gt_to_pred.mean().item()

    fscores = {}
    for t in thresholds:
        precision = (d_pred_to_gt < t).float().mean().item()
        recall = (d_gt_to_pred < t).float().mean().item()
        f1 = 2 * precision * recall / (precision + recall + 1e-8)
        fscores[t] = {"precision": precision, "recall": recall, "f1": f1}

    return chamfer, fscores


def align_icp(pred_pts, gt_pts):
    """Alinea pred hacia gt con ICP (rotación + traslación + escala).
    Devuelve los puntos de pred ya alineados y la transformación estimada."""
    solution = iterative_closest_point(pred_pts, gt_pts, allow_reflection=False)
    return solution.Xt, solution.RTs  # Xt: pred alineado, RTs: (R, T, s)


def apply_similarity_to_verts(verts, RTs):
    """Aplica la transformación estimada por ICP a un set completo de vértices
    (no solo a los puntos muestreados), para poder reconstruir la malla alineada."""
    R, T, s = RTs.R, RTs.T, RTs.s
    verts = verts.unsqueeze(0)  # (1, V, 3)
    verts_aligned = s[:, None, None] * torch.bmm(verts, R) + T[:, None, :]
    return verts_aligned.squeeze(0)


def volumetric_iou(mesh_pred_tm, mesh_gt_tm, resolution=64):
    """
    Calcula el IoU volumétrico convirtiendo las mallas de Trimesh a Open3D
    y generando Grillas de Vóxeles de forma nativa y ultra rápida.
    """
    # 1. Convertir Trimesh a Open3D
    o3d_pred = o3d.geometry.TriangleMesh()
    o3d_pred.vertices = o3d.utility.Vector3dVector(mesh_pred_tm.vertices)
    o3d_pred.triangles = o3d.utility.Vector3iVector(mesh_pred_tm.faces)

    o3d_gt = o3d.geometry.TriangleMesh()
    o3d_gt.vertices = o3d.utility.Vector3dVector(mesh_gt_tm.vertices)
    o3d_gt.triangles = o3d.utility.Vector3iVector(mesh_gt_tm.faces)

    # 2. Definir el tamaño del vóxel basado en la resolución deseada
    # Como los modelos están normalizados en una esfera unitaria, el tamaño total es ~2.0
    voxel_size = 2.0 / resolution

    # 3. Voxelizar de forma eficiente en C++
    voxel_grid_pred = o3d.geometry.VoxelGrid.create_from_triangle_mesh(o3d_pred, voxel_size=voxel_size)
    voxel_grid_gt = o3d.geometry.VoxelGrid.create_from_triangle_mesh(o3d_gt, voxel_size=voxel_size)

    # 4. Obtener las coordenadas de los vóxeles ocupados
    voxels_pred = set(tuple(v.grid_index) for v in voxel_grid_pred.get_voxels())
    voxels_gt = set(tuple(v.grid_index) for v in voxel_grid_gt.get_voxels())

    # 5. Calcular Intersección y Unión usando operaciones de conjuntos de Python (muy rápido)
    inter = len(voxels_pred.intersection(voxels_gt))
    union = len(voxels_pred.union(voxels_gt))

    return float(inter) / float(union) if union > 0 else 0.0


# ---------------------------------------------------------------------------
# Pipeline por par (gt, pred)
# ---------------------------------------------------------------------------
@torch.no_grad()
def evaluate_pair(gt_path, pred_path, device, n_points=DEFAULT_N_POINTS,
                   thresholds=DEFAULT_THRESHOLDS, voxel_res=DEFAULT_VOXEL_RES):

    mesh_gt = normalize_mesh(load_mesh_p3d(gt_path, device))
    mesh_pred = normalize_mesh(load_mesh_p3d(pred_path, device))

    pts_gt = sample_points_from_meshes(mesh_gt, num_samples=n_points)
    pts_pred = sample_points_from_meshes(mesh_pred, num_samples=n_points)

    pts_pred_aligned, RTs = align_icp(pts_pred, pts_gt)

    chamfer, fscores = chamfer_and_fscore(pts_pred_aligned, pts_gt, thresholds)

    # Aplicar la misma transformación a la malla completa (no solo a los puntos
    # muestreados) para poder calcular el IoU volumétrico sobre la superficie real.
    verts_pred_aligned = apply_similarity_to_verts(mesh_pred.verts_packed(), RTs)
    mesh_pred_aligned = mesh_pred.update_padded(verts_pred_aligned[None])

    tm_pred = mesh_to_trimesh(mesh_pred_aligned)
    tm_gt = mesh_to_trimesh(mesh_gt)

    try:
        iou = volumetric_iou(tm_pred, tm_gt, resolution=voxel_res)
    except Exception as e:
        print(f"  [aviso] no se pudo calcular IoU volumétrico ({e}); "
              f"probablemente la malla no es watertight.")
        iou = None

    result = {"chamfer": chamfer, "iou_volumetric": iou}
    for t, vals in fscores.items():
        result[f"fscore@{t}"] = vals["f1"]
        result[f"precision@{t}"] = vals["precision"]
        result[f"recall@{t}"] = vals["recall"]

    return result


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def print_result(name, result):
    print(f"\n=== {name} ===")
    print(f"  Chamfer Distance : {result['chamfer']:.6f}")
    if result["iou_volumetric"] is not None:
        print(f"  IoU volumétrico  : {result['iou_volumetric']:.4f}")
    else:
        print(f"  IoU volumétrico  : N/A")
    for t in DEFAULT_THRESHOLDS:
        print(f"  F-score@{t:<5} : {result[f'fscore@{t}']:.4f} "
              f"(P={result[f'precision@{t}']:.4f}, R={result[f'recall@{t}']:.4f})")


def save_csv(results, out_path="eval_3d_results.csv"):
    if not results:
        return
    fieldnames = ["name"] + [k for k in results[0].keys() if k != "name"]
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in results:
            writer.writerow(r)
    print(f"\nResultados guardados en: {out_path}")


def main(EXPERIMENTS, type_rec, DEFAULT_N_POINTS, DEFAULT_VOXEL_RES):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    all_results = []
    for exp in EXPERIMENTS:
        print(f"\nCalculando metricas: {exp["name"]} - {type_rec}")
        if not (os.path.exists(exp["gt"]) and os.path.exists(exp["pred"])):
            print(f"[saltando] {exp['name']}: no se encontró gt o pred "
            f"({exp['gt']} / {exp['pred']})")
            continue
        result = evaluate_pair(exp["gt"], exp["pred"], device, n_points=DEFAULT_N_POINTS, voxel_res=DEFAULT_VOXEL_RES)
        print_result(exp["name"], result)
        result["name"] = exp["name"]
        all_results.append(result)

        torch.cuda.empty_cache()
        gc.collect()

    save_csv(all_results, f"/content/{type_rec}_metrics.csv")


In [14]:
# ---------------------------------------------------------------------------
# Lista para correr todos tus experimentos de una sola vez.
# "gt" es el modelo original, "pred" es el .obj generado por el pipeline.
# ---------------------------------------------------------------------------
EXPERIMENTS_VOXEL = [
    {"name": "car_1",   "gt": "gt_models/car.obj",   "pred": "voxel/car_1/sample_0_output.obj"},
    {"name": "car_2",   "gt": "gt_models/car.obj",   "pred": "voxel/car_2/sample_0_output.obj"},
    {"name": "car_3",   "gt": "gt_models/car.obj",   "pred": "voxel/car_3/sample_0_output.obj"},
    {"name": "rifle_1", "gt": "gt_models/rifle.obj", "pred": "voxel/rifle_1/sample_0_output.obj"},
    {"name": "rifle_2", "gt": "gt_models/rifle.obj", "pred": "voxel/rifle_2/sample_0_output.obj"},
    {"name": "rifle_3", "gt": "gt_models/rifle.obj", "pred": "voxel/rifle_3/sample_0_output.obj"},
]

EXPERIMENTS_MESH = [
    {"name": "car_1",   "gt": "gt_models/car.obj",   "pred": "mesh/car_1/sample_0_output.obj"},
    {"name": "car_2",   "gt": "gt_models/car.obj",   "pred": "mesh/car_2/sample_0_output.obj"},
    {"name": "car_3",   "gt": "gt_models/car.obj",   "pred": "mesh/car_3/sample_0_output.obj"},
    {"name": "rifle_1", "gt": "gt_models/rifle.obj", "pred": "mesh/rifle_1/sample_0_output.obj"},
    {"name": "rifle_2", "gt": "gt_models/rifle.obj", "pred": "mesh/rifle_2/sample_0_output.obj"},
    {"name": "rifle_3", "gt": "gt_models/rifle.obj", "pred": "mesh/rifle_3/sample_0_output.obj"},
]

In [15]:
main(EXPERIMENTS_VOXEL, "voxel", DEFAULT_N_POINTS, DEFAULT_VOXEL_RES)

Device: cuda:0

Calculando metricas: car_1 - voxel

=== car_1 ===
  Chamfer Distance : 0.350641
  IoU volumétrico  : 0.0329
  F-score@0.01  : 0.0094 (P=0.0088, R=0.0101)
  F-score@0.02  : 0.0425 (P=0.0380, R=0.0483)
  F-score@0.05  : 0.1325 (P=0.1089, R=0.1693)

Calculando metricas: car_2 - voxel

=== car_2 ===
  Chamfer Distance : 0.300848
  IoU volumétrico  : 0.0269
  F-score@0.01  : 0.0118 (P=0.0119, R=0.0118)
  F-score@0.02  : 0.0562 (P=0.0584, R=0.0542)
  F-score@0.05  : 0.1952 (P=0.1943, R=0.1962)

Calculando metricas: car_3 - voxel

=== car_3 ===
  Chamfer Distance : 0.276455
  IoU volumétrico  : 0.0386
  F-score@0.01  : 0.0170 (P=0.0169, R=0.0171)
  F-score@0.02  : 0.0724 (P=0.0712, R=0.0737)
  F-score@0.05  : 0.2293 (P=0.2312, R=0.2274)

Calculando metricas: rifle_1 - voxel

=== rifle_1 ===
  Chamfer Distance : 0.270355
  IoU volumétrico  : 0.0061
  F-score@0.01  : 0.0199 (P=0.0143, R=0.0325)
  F-score@0.02  : 0.0528 (P=0.0311, R=0.1753)
  F-score@0.05  : 0.1377 (P=0.0792, R=0

In [16]:
main(EXPERIMENTS_MESH, "mesh", DEFAULT_N_POINTS, DEFAULT_VOXEL_RES)

Device: cuda:0

Calculando metricas: car_1 - mesh

=== car_1 ===
  Chamfer Distance : 0.366105
  IoU volumétrico  : 0.0287
  F-score@0.01  : 0.0054 (P=0.0050, R=0.0059)
  F-score@0.02  : 0.0264 (P=0.0237, R=0.0299)
  F-score@0.05  : 0.1073 (P=0.0884, R=0.1364)

Calculando metricas: car_2 - mesh

=== car_2 ===
  Chamfer Distance : 0.342401
  IoU volumétrico  : 0.0173
  F-score@0.01  : 0.0072 (P=0.0070, R=0.0075)
  F-score@0.02  : 0.0357 (P=0.0341, R=0.0375)
  F-score@0.05  : 0.1427 (P=0.1219, R=0.1720)

Calculando metricas: car_3 - mesh

=== car_3 ===
  Chamfer Distance : 0.406776
  IoU volumétrico  : 0.0165
  F-score@0.01  : 0.0080 (P=0.0075, R=0.0085)
  F-score@0.02  : 0.0441 (P=0.0403, R=0.0488)
  F-score@0.05  : 0.1328 (P=0.1097, R=0.1682)

Calculando metricas: rifle_1 - mesh

=== rifle_1 ===
  Chamfer Distance : 0.508373
  IoU volumétrico  : 0.0070
  F-score@0.01  : 0.0042 (P=0.0032, R=0.0063)
  F-score@0.02  : 0.0109 (P=0.0067, R=0.0295)
  F-score@0.05  : 0.0300 (P=0.0177, R=0.097